In [10]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import HumanMessage, BaseMessage
from typing import Annotated, TypedDict, Literal
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph.message import add_messages
from pydantic import BaseModel
from dotenv import load_dotenv
import operator
import os

api_key = os.getenv("GOOGLE_API_KEY")

In [ ]:
#required models
geimi = ChatGoogleGenerativeAI(
    model = "gemini-2.5-pro",
    max_retries=2,
    max_tokens=1024,
    temperature=0.7,
    google_api_key = api_key
)

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
)

mistral = ChatHuggingFace(llm=llm)

In [11]:
class ChateState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [13]:
def chat(state: ChateState) -> dict:
    response = mistral.invoke(state['messages'])
    return {'messages': response}

In [14]:
graph = StateGraph(ChateState)

graph.add_node('chat', chat)

graph.add_edge(START, 'chat')
graph.add_edge('chat', END)

chatbot = graph.compile()

In [15]:
chatbot.get_graph().print_ascii()

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +------+     
  | chat |     
  +------+     
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [16]:
intial_sate = {
    'messages': HumanMessage(content="Which is Capital of Maharastra")
}

In [17]:
final_state = chatbot.invoke(intial_sate)

In [18]:
final_state

{'messages': [HumanMessage(content='Which is Capital of Maharastra', additional_kwargs={}, response_metadata={}, id='a2092b82-0f95-40b8-a693-d62bc2fb16ee'),
  AIMessage(content=' The capital city of Maharashtra, a state in western India, is Mumbai (also known as Bombay). It is the financial capital of India and a major global power city. Mumbai is the most populous city in Maharashtra and the second most populous city in India. The administrative functions of Maharashtra are carried out in Mumbai from the Maharashtra State Secretariat located in South Mumbai.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 13, 'total_tokens': 113}, 'model_name': 'mistralai/Mistral-7B-Instruct-v0.2', 'system_fingerprint': '', 'finish_reason': 'stop', 'logprobs': None}, id='run--5fb0e80e-b184-4b4d-a54d-3c6956e93da5-0', usage_metadata={'input_tokens': 13, 'output_tokens': 100, 'total_tokens': 113})]}